# 08 — Caractériser un rythme géométriquement (Fourier / nombres complexes)

Idée : fixer **le cercle = un cycle**, mettre tous les rythmes sur **la même période**, et les comparer par des **angles** et des **nombres complexes**.

C'est l'**analyse de Fourier des rythmes** (théorie mathématique de la musique, Amiot/Milne) :
- une frappe à la position $\varphi$ du cycle = un point $e^{i\varphi}$ sur le cercle unité ;
- un rythme = quelques **coefficients complexes** $M_n = \sum_k m_k\,e^{-2i\pi nk/12}$ ;
- $|M_n|$ = force de la périodicité $n$ (la **forme**, invariante) ; l'angle de $M_n$ = le **placement** ;
- comparer deux rythmes = comparer ces vecteurs (distance, angle).

→ chaque rythme devient un **point** dans un « espace des rythmes ».

## 0. Imports & motif de chaque rythme

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.manifold import MDS
import utils_V2 as u
df = pd.read_csv('output/dunun_frappes_V2.csv'); d1 = df[df['code']=='D1']
dd = u.list_dunun(); COL = dict(zip(u.PIECES, ['#4a8cff', '#e0952e', '#2fbf6e']))

def motif(p):
    return np.mean([pr.values for t in dd.query('piece==@p and code=="D1"')['take'].unique()
                    if (pr := u.subdivision_profile(p, int(t), 'D1')) is not None], axis=0)
MOT = {p: motif(p) for p in u.PIECES}
M   = {p: np.fft.fft(MOT[p]) for p in u.PIECES}   # coefficients complexes

## 1. Les rythmes sur le cercle unité
Même période pour tous. Angle = position réelle (avec micro-timing), taille = densité de frappe.

In [ ]:
def phase(p):
    s = d1[d1['piece']==p]
    return {sd: s[s['SD']==sd]['rel'].mean()*2*np.pi for sd in range(1,13) if (s['SD']==sd).sum() > 20}
ax = plt.subplot(111, projection='polar'); ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
for p in u.PIECES:
    for sd, th in phase(p).items():
        ax.scatter(th, 1, s=40+280*MOT[p][sd-1], color=COL[p], alpha=.65, edgecolors='none')
ax.scatter([],[],color='#4a8cff',label='Manjanin'); ax.scatter([],[],color='#e0952e',label='Maraka'); ax.scatter([],[],color='#2fbf6e',label='Woloso')
ax.set(yticklabels=[], ylim=(0,1.2)); ax.set_xticks(np.arange(12)/12*2*np.pi); ax.set_xticklabels(range(1,13))
ax.legend(loc='center'); plt.title('Les 3 rythmes sur le cercle unité (1 cycle)'); plt.show()

## 2. L'empreinte de Fourier
$|M_n|$ = force de la périodicité $n$. Chaque rythme a un spectre distinct.

In [ ]:
x = np.arange(1,7); w = .25
plt.figure(figsize=(9,4))
for i, p in enumerate(u.PIECES):
    plt.bar(x+(i-1)*w, [abs(M[p][n]) for n in range(1,7)], w, color=COL[p], label=p)
plt.xlabel('périodicité n'); plt.ylabel('|Mₙ|'); plt.xticks(x); plt.legend()
plt.title('Empreinte de Fourier du dunun'); plt.show()
print('Manjanin culmine à n=6, Maraka à n=5, Woloso à n=4 (sa structure alternée).')

## 3. Distances entre rythmes
Dans l'espace des empreintes $[|M_1|..|M_6|]$ (invariant au choix du temps 1).

In [ ]:
fp = np.array([[abs(M[p][n]) for n in range(1,7)] for p in u.PIECES])
D = np.array([[np.linalg.norm(fp[i]-fp[j]) for j in range(len(u.PIECES))] for i in range(len(u.PIECES))])
display(pd.DataFrame(np.round(D,2), index=u.PIECES, columns=u.PIECES))
plt.imshow(D, cmap='magma_r'); plt.colorbar(label='distance')
plt.xticks(range(3), u.PIECES); plt.yticks(range(3), u.PIECES); plt.title('Distances entre rythmes'); plt.show()

## 4. La carte des rythmes (2D)
Projection (MDS) qui respecte les distances : chaque rythme = un point. **Extensible** — on peut y ajouter d'autres rythmes (YouTube, etc.) pour peupler la carte.

In [ ]:
xy = MDS(n_components=2, dissimilarity='precomputed', random_state=0, normalized_stress='auto').fit_transform(D)
plt.figure(figsize=(6,6))
for (a,b), p in zip(xy, u.PIECES):
    plt.scatter(a, b, s=300, color=COL[p]); plt.annotate(p, (a,b), textcoords='offset points', xytext=(0,12), ha='center')
plt.title('Carte des rythmes (espace de Fourier)'); plt.axhline(0,color='#eee'); plt.axvline(0,color='#eee'); plt.axis('equal'); plt.show()

## 5. « L'équation du rythme »
On garde seulement les **k plus gros coefficients** et on reconstruit le motif (transformée inverse). Peu de coefficients suffisent → le rythme *est* une petite somme d'exponentielles complexes.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a, p in zip(ax, u.PIECES):
    m = MOT[p]; Mp = np.fft.fft(m)
    a.step(range(1,13), m, where='mid', color='k', lw=2, label='vrai motif')
    for k, c in [(2,'#bbb'), (4,COL[p])]:
        idx = np.argsort(-np.abs(Mp)); keep = np.zeros(12, complex); keep[idx[:k*2]] = Mp[idx[:k*2]]
        rec = np.fft.ifft(keep).real
        err = np.linalg.norm(rec-m)/np.linalg.norm(m)*100
        a.step(range(1,13), rec, where='mid', color=c, lw=1.6, ls='--', label=f'{k} coeffs ({err:.0f}%)')
    a.set(title=p, xlabel='subdivision', xticks=range(1,13)); a.legend(fontsize=8)
plt.suptitle('Reconstruire le rythme avec quelques coefficients complexes'); plt.tight_layout(); plt.show()

## Bilan
- Chaque rythme est **caractérisé par quelques nombres complexes** (son empreinte de Fourier) — l'« équation » que tu cherchais.
- On mesure **distances et angles** entre rythmes, et on les place sur une **carte 2D**.
- **4 coefficients** reconstruisent un rythme à ~10 % près (6 = exact) → une représentation ultra-compacte.
- **Visualisation interactive** : `python boussole_rythmes_V2.py` génère la Boussole des rythmes, avec les équations complexes fixes de D1/D2.
- **Prochaine force** : ajouter d'autres rythmes (Djansa, Gede… via la séparation du dunun) pour **peupler la carte** et rendre les distances vraiment parlantes.

Note : ici on caractérise le **motif** (le *où*). On peut enrichir l'empreinte avec le **micro-timing** (phase fine) et la **densité** pour une signature encore plus complète.